## If / Else Agent

This notebook mirrors `src/01_if_else_agent.py` so you can:

- Tweak the grid-based rules and re-run without leaving Jupyter.
- Inspect **offline** recordings (parse observations, print sensors, plot a frame).
- Then run the **live** game loop against the Godot environment.

Observation layout: `observation[:-8]` is the flattened grid; `observation[-8:]` are extra sensors (see `00_data_exploration.ipynb`). Grid indexing is `[row, col]`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
from pathlib import Path

from utils import setup_environment, load_observations_by_session
from processing.frame_visualizer import FrameVisualizer

In [ ]:
# Build once per kernel session (avoids recreating the visualizer every step).
visualizer = FrameVisualizer()


def agent_brain(observation, step_count):
    parsed = visualizer.parse_observation(observation)
    grid = parsed["grid"]

    # Example heuristic: jump when you see a wall on your right.
    # First index is row, second is column.
    # You need to press jump on the first frame to start walking.
    if grid[3, 6] != 0 or step_count == 0:
        return [1]  # jump

    return [0]  # do nothing

### Prototype on recorded data

Point `RECORDING` at any `.jsonl` session file under `data/`. Adjust `FRAME_INDEX` to step through frames and check whether your `if` conditions match what you expect before running the live env.

In [ ]:
RECORDING = Path("../data/cheatsheet/session_1/demo.jsonl")
FRAME_INDEX = 0

by_session = load_observations_by_session(RECORDING)
session_id = next(iter(by_session))
frames = by_session[session_id]
frame = frames[FRAME_INDEX]
state = frame["state"]
human_action = int(frame["action"])

print(visualizer.format_observation_text(state, include_grid=False))
print("human action:", human_action, "| heuristic (step=0):", agent_brain(state, 0)[0])
print("human action:", human_action, "| heuristic (step=5):", agent_brain(state, 5)[0])

info = visualizer.extract_frame_info(frame)
visualizer.plot_frame_info(info, session_id=session_id, frame_index=FRAME_INDEX, action=human_action)

### Run the live environment

Requires Godot / env paths from `src/.config` (same as running `python src/01_if_else_agent.py`). Interrupt the kernel to stop early.

In [ ]:
env = setup_environment()
obs = env.reset()
nb_agents = len(obs["obs"])

step_count = 0
while True:
    actions = [agent_brain(obs["obs"][i], step_count) for i in range(nb_agents)]
    actions = np.array(actions, dtype=np.int64)
    obs, reward, done, info = env.step(actions)
    if any(done):
        break
    step_count += 1

env.close()